In [ ]:
# tamserve-boxes (Rust): YOLOX-Nano on gray frames, on the CPU through OpenVINO, for many cameras.
# One cell: install Rust and GStreamer's development files if needed, build the crate, make the 1-channel model, check the Rust port against
# the Python version (model, detections, tracking), and compare their CPU and memory. Colab's CPU runtime is enough.
import json, os, shutil, subprocess, sys, urllib.request, hashlib
from pathlib import Path

RUST_GIT = os.environ.get("TSB_RUST_GIT", "https://github.com/dhaneshkk/tamserve-boxes")    # this crate
PYTHON_GIT = os.environ.get("TSB_PYTHON_GIT", "https://github.com/dhaneshkk/tamserve")       # the Python version it is checked against
WORK = Path("/content" if Path("/content").exists() else Path.cwd()) / "rust_boxes"
WORK.mkdir(parents=True, exist_ok=True)
CRATE, REPO = WORK / "tamserve-boxes", WORK / "tamserve"
sh = lambda *a, **k: subprocess.run(list(a), check=True, **k)
for url, path in ((RUST_GIT, CRATE), (PYTHON_GIT, REPO)):
    if not path.exists():
        sh("git", "clone", "-q", "--depth", "1", url, str(path))
sh(sys.executable, "-m", "pip", "install", "-q", "openvino>=2024.0", "onnx", "opencv-python-headless", "psutil")
cargo = shutil.which("cargo") or str(Path.home() / ".cargo/bin/cargo")
if not Path(cargo).exists():
    sh("bash", "-c", "curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    cargo = str(Path.home() / ".cargo/bin/cargo")

# OpenVINO's C library from the pip wheel, linked under the plain name the Rust bindings look for.
import openvino
OV = Path(openvino.__file__).parent / "libs"
LINK = WORK / "ovlib"; LINK.mkdir(exist_ok=True)
lib = next(OV.glob("libopenvino_c.so.*"))
(LINK / "libopenvino_c.so").unlink(missing_ok=True); (LINK / "libopenvino_c.so").symlink_to(lib)
env = {**os.environ, "LD_LIBRARY_PATH": f"{LINK}:{OV}:" + os.environ.get("LD_LIBRARY_PATH", "")}
have_gst = shutil.which("pkg-config") and subprocess.run(["pkg-config", "--exists", "gstreamer-app-1.0"]).returncode == 0
if not have_gst and shutil.which("apt-get"):  # GStreamer's development files, which the crate builds against
    sh("bash", "-c", "apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y pkg-config "
       "libgstreamer1.0-dev libgstreamer-plugins-base1.0-dev gstreamer1.0-plugins-good gstreamer1.0-libav > /dev/null")
sh(cargo, "build", "--release", "-q", "--examples", "--bins", cwd=CRATE, env=env)
BIN = CRATE / "target/release/tamserve-boxes"

def fetch(url, sha256):
    path = WORK / url.rsplit("/", 1)[1]
    if not path.exists():
        data = urllib.request.urlopen(url, timeout=120).read()
        assert hashlib.sha256(data).hexdigest() == sha256, f"{url} is not the pinned file"
        path.write_bytes(data)
    return path
nano = fetch("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx",
             "c789161ed43c8269fcd4e67c67eeeb4e80c622da2eb296a20bc6007bd18a0b7d")
dog = fetch("https://raw.githubusercontent.com/Megvii-BaseDetection/YOLOX/6ddff4824372906469a7fae2dc3206c7aa4bbaee/assets/dog.jpg",
            "5a9522051c3cec2bbd2f6323fccba32e8fbf3ddcc2b3e2fd46b04c720bc6f866")

sys.path.insert(0, str(REPO / "src"))
import cv2, numpy as np, onnx
from onnx import numpy_helper
import tamserve.boxes as B

# 1. The 1-channel model, made by Rust and by Python: the same weights.
sh(str(BIN), "fold", str(nano), str(WORK / "rust_gray.onnx"), env=env)
B.gray_model(nano, WORK / "py_gray.onnx")
w = lambda p: {t.name: numpy_helper.to_array(t) for t in onnx.load(str(p)).graph.initializer}
wr, wp = w(WORK / "rust_gray.onnx"), w(WORK / "py_gray.onnx")
print("1. fold: weights that differ between Rust and Python:", [k for k in wp if not np.array_equal(wr[k], wp[k])] or "none")

# 2. Detections on gray pictures of several sizes, float32.
gray = cv2.imread(str(dog), cv2.IMREAD_GRAYSCALE)
pics = []
for w_, h_ in ((768, 576), (1280, 720), (1920, 1080), (352, 288), (417, 1001)):
    p = WORK / f"dog_{w_}x{h_}.pgm"; cv2.imwrite(str(p), cv2.resize(gray, (w_, h_), interpolation=cv2.INTER_CUBIC)); pics.append(p)
out = subprocess.run([str(BIN), "detect", "--model", str(WORK / "rust_gray.onnx"), "--precision", "f32", *map(str, pics)],
                     capture_output=True, text=True, check=True, env=env).stdout.splitlines()
det = B.Detector(WORK / "py_gray.onnx", precision="f32")
worst, differ = 0.0, 0
for line, p in zip(out, pics):
    rs = json.loads(line)["finds"]; pb, ps, pl = det([cv2.imread(str(p), 0)])[0]
    differ += len(rs) != len(ps) or any(B.COCO[l] != f["label"] for l, f in zip(pl, rs))
    worst = max([worst] + [abs(float(s) - f["score"]) for s, f in zip(ps, rs)])
print(f"2. detect, {len(pics)} picture sizes: pictures whose finds differ: {differ}; largest score difference {worst:.1e}")

# 3. Tracking, 3 cameras x 60 frames (the dog is covered for a while on two of them): the same ids on every frame.
W, H, N = 960, 640, 60
def video(seed, covered):
    rng = np.random.default_rng(seed); bg = cv2.GaussianBlur(rng.integers(60, 200, (H, W), dtype=np.uint8), (0, 0), 9)
    fs = []
    for k in range(N):
        f = bg.copy(); x = (10 + 3 * k + 25 * seed) % (W - gray.shape[1]); f[40:40 + gray.shape[0], x:x + gray.shape[1]] = gray
        if k in covered: f[240:600, x + 110:x + 350] = 128
        fs.append(f)
    return np.stack(fs)
vids = [video(0, range(20, 27)), video(1, ()), video(2, range(35, 50))]
for i, v in enumerate(vids): v.tofile(WORK / f"cam{i}.raw")
rs = [json.loads(l) for l in subprocess.run([str(CRATE / "target/release/examples/track_frames"), str(WORK / "rust_gray.onnx"),
      str(W), str(H), "3", *[str(WORK / f"cam{i}.raw") for i in range(3)]], capture_output=True, text=True, check=True, env=env).stdout.splitlines()]
tr = B.BoxTracker(B.Detector(WORK / "py_gray.onnx", threads=2, precision="f32", classes=("dog", "bicycle", "car")), look_every=3)
differ = 0
for k in range(N):
    o = tr.step({c: vids[c][k] for c in range(3)})
    differ += sum(sorted((i, f.label) for i, f in o[c].items()) != sorted((x[0], x[1]) for x in rs[k][c]) for c in range(3))
print(f"3. track, 3 cameras x {N} frames, look_every=3: frames where Rust and Python ids differ: {differ}")

# 4. CPU and memory, 4 cameras at 10 frames/s: Rust and Python, one shared process each.
cv2.imwrite(str(WORK / "dog.pgm"), gray)
r = json.loads(subprocess.run([str(BIN), "bench", "--model", str(WORK / "rust_gray.onnx"), "--image", str(WORK / "dog.pgm"),
                               "--cameras", "4", "--fps", "10", "--seconds", "15", "--threads", "1"],
                              capture_output=True, text=True, check=True, env=env).stdout)
p = json.loads(subprocess.run([sys.executable, str(REPO / "bench/boxes_cpu.py"), "--model", str(WORK / "py_gray.onnx"), "--image", str(dog),
                               "--cameras", "4", "--fps", "10", "--seconds", "15", "--threads", "1"],
                              capture_output=True, text=True, check=True, env={**env, "PYTHONPATH": str(REPO / "src")}).stdout)["shared"]
print(f"4. 4 cameras x 10 fps, one process: Rust {r['cpu_cores']} cores, {r['memory_mb']} MB | Python {p['cpu_cores']} cores, {p['memory_mb']} MB"
      " (both hold ~27 MB of frames per camera)")
